# NN Grid Research

## Setup

In [ ]:
import keras
import time
import os
os.environ["KERAS_BACKEND"] = "torch"

%load_ext autoreload
%autoreload 2
print(keras.backend.backend())

## LUT Upload

In [ ]:
from src.lut.lut import Lut

LUT_PATH = "models/luts/W_all/p3/lut_p3_W_all.nc"
VALIDATION_LUT_PATH = "models/luts/W_all/p3/lut_p3_validation.nc"

my_lut = Lut.load(LUT_PATH)
my_lut.print_info()

validation_lut = Lut.load(VALIDATION_LUT_PATH)
validation_lut.print_info()

## Grid Search Algorithm
This section trains several models and saves it into a list.
The cell execute every combination.

This section takes several minutes to be computed.

In [ ]:
from src.nn.nn_generator import NnGenerator

NUM_LAYERS_TO_TRY = [2]
NUM_UNITS_TO_TRY = [32, 64]
HIDDEN_FUNCTIONS_TO_TRY = ["relu"]
OUTPUT_FUNCTIONS_TO_TRY = ["linear"]
SEED = 43

EPOCHS = 60
BATCH_SIZE = 512
LEARNING_RATE = 3e-3
LR_FINAL_FACTOR = 1e-3

DIR_GS = "models/nn/W_all/grid_search"
W_TO_VISUALIZE = [0.0, 0.2, 0.5, 0.8, 1.0]
VELOCITY_TO_VISUALIZE = [0.0, 600.0]

n_combinations = (len(NUM_LAYERS_TO_TRY) * len(NUM_UNITS_TO_TRY)
                  * len(HIDDEN_FUNCTIONS_TO_TRY) * len(OUTPUT_FUNCTIONS_TO_TRY))
print(f"Grid Search: {n_combinations} combinations...")
start_total = time.time()

nn_generator = NnGenerator(my_lut) # training data prepared once for all the networks

for n_layers in NUM_LAYERS_TO_TRY:
    for n_units in NUM_UNITS_TO_TRY:
        for hidden_fn in HIDDEN_FUNCTIONS_TO_TRY:
            for out_fn in OUTPUT_FUNCTIONS_TO_TRY:

                model_name = f"{n_layers}L_{n_units}U_{EPOCHS}E_{hidden_fn}_{out_fn}_cosdecay"
                print("\n" + "=" * 60)
                print(f"TRAINING: {model_name}")
                print("=" * 60)
                start_train = time.time()

                # --- directories ---
                dir_nn = os.path.join(DIR_GS, model_name)
                dir_plots = os.path.join(dir_nn, "plots")
                dir_heatmaps = os.path.join(dir_plots, "heatmaps")

                # --- NN SETUP ---
                nn_generator.define_structure_nn(num_hidden_layers=n_layers,
                                                 hidden_units=n_units,
                                                 hidden_function=hidden_fn,
                                                 output_function=out_fn,
                                                 seed=SEED)

                # --- NN TRAINING ---
                nn_generator.train_nn(nn_name=model_name,
                                      epochs=EPOCHS,
                                      batch_size=BATCH_SIZE,
                                      learning_rate=LEARNING_RATE,
                                      lr_final_factor=LR_FINAL_FACTOR,
                                      verbose=0,
                                      print_result=False)
                my_nn = nn_generator.nn

                # --- Validation ---
                my_nn.validate(validation_lut)

                # --- Save network ---
                my_nn.save(dir_nn, lut=my_lut)

                # --- Save plots ---
                my_nn.plot_structure(save_path=os.path.join(dir_plots, f"{model_name}_graph.png"), show=False)
                my_nn.plot_loss(save_path=os.path.join(dir_plots, f"{model_name}_loss.png"), show=False)
                my_nn.plot_learning_rate(save_path=os.path.join(dir_plots, f"{model_name}_learning_rate.png"),
                                         show=False)
                my_nn.plot_parity(my_lut, save_path=os.path.join(dir_plots, f"{model_name}_parity_all.png"),
                                  show=False)
                my_nn.plot_parity(validation_lut, show=False,
                                  save_path=os.path.join(dir_plots, f"{model_name}_parity_validation.png"))

                for vel_k in VELOCITY_TO_VISUALIZE:
                    for W_k in W_TO_VISUALIZE:
                        tag = f"W_{W_k:g}_v_{vel_k:g}"     # slice of the heatmaps, in the file names
                        my_nn.plot_heatmaps_activations(my_lut, W_k, vel_k, show=False,
                            save_path=os.path.join(dir_heatmaps, f"{model_name}_hmps_u_{tag}.png"))
                        my_nn.plot_heatmaps_error(my_lut, W_k, vel_k, show=False,
                            save_path=os.path.join(dir_heatmaps, f"{model_name}_hmps_error_{tag}.png"))
                        my_nn.plot_heatmap_torque_error(my_lut, W_k, vel_k, show=False,
                            save_path=os.path.join(dir_heatmaps, f"{model_name}_hmp_torque_error_{tag}.png"))

                print(f"Completed in {time.time() - start_train:.1f} s.")
                keras.backend.clear_session()

total_time = time.time() - start_total
print("\n" + "★" * 60)
print(f"GRID SEARCH COMPLETED IN {total_time / 60:.1f} MINUTES!")
print("★" * 60)